# sisl · SIESTA matrices and visualization

This tutorial introduces how to analyze SIESTA calculation results using the Python package sisl. Starting from a simple bulk BN system, we explore the Hamiltonian (H), overlap (S), and density (DM) matrices and their representations in an atomic orbital basis.

We then use these matrices to calculate and visualize electronic properties, including band structures, band folding, and real-space electron densities.

Run the cells in order in a **Colab CPU runtime**. The small SZP basis and sampling settings are for learning the workflow; quantitative calculations require separate convergence tests.


## 1. Setup

Install **SIESTA 5.4.2 (serial)** and **sisl 0.16.4**. SIESTA runs as a separate executable; Python cells use the Colab kernel. Installation takes a few minutes.

In [ ]:
%%bash
set -euo pipefail
mkdir -p /content/mamba
curl -Ls --fail https://micro.mamba.pm/api/micromamba/linux-64/latest | tar -xj -C /content/mamba bin/micromamba
/content/mamba/bin/micromamba create -y -r /content/mamba -p /content/siesta-env \
    -c conda-forge --override-channels 'siesta=5.4.2=nompi_*'
/content/siesta-env/bin/siesta --version

In [ ]:
%pip install -q "sisl==0.16.4" matplotlib netCDF4

import os
os.environ["OMP_NUM_THREADS"] = "2"
os.environ["OPENBLAS_NUM_THREADS"] = "1"

from pathlib import Path
from urllib.request import urlretrieve
import subprocess
import numpy as np
import matplotlib.pyplot as plt
import sisl as si

print("sisl:", si.__version__)

## 2. Run BN with SIESTA

`RUN.fdf` sets the SZP basis, k mesh, and SCF options; it includes the lattice and four atomic positions from `STRUCT.fdf`. Keep `B.psf` and `N.psf` in the same directory.

The output options used here are `SaveHS` for H/S, `Write.DM` for DM, and `WriteIonXML` for the numerical orbitals needed on real-space grids.

In [ ]:
work = Path("/content/bn-matrices")
work.mkdir(parents=True, exist_ok=True)
base = "https://raw.githubusercontent.com/YHKlab-RGLee/tutorial/main/sisl/01/src/inputs"
for name in ["RUN.fdf", "STRUCT.fdf", "B.psf", "N.psf"]:
    urlretrieve(f"{base}/{name}", work / name)
os.chdir(work)

fdf = si.get_sile("RUN.fdf")
structure = fdf.read_geometry()
print(f"Structure: {structure.na} atoms, two BN layers")
print(Path("RUN.fdf").read_text())

In [ ]:
with Path("siesta.out").open("w") as output:
    subprocess.run(["/content/siesta-env/bin/siesta", "RUN.fdf"],
                   stdout=output, stderr=subprocess.STDOUT, check=True)

log = Path("siesta.out").read_text()
if "SCF Convergence" not in log or "SCF_NOT_CONV" in log:
    raise RuntimeError("SCF did not converge; read siesta.out")
required = ["BN.HSX", "BN.DM", "B.ion.xml", "N.ion.xml"]
missing = [name for name in required if not Path(name).is_file()]
if missing:
    raise FileNotFoundError(f"Missing outputs: {missing}; read siesta.out")
print("SCF complete. Ready to read H, S, and DM.")

## 3. Read H, S, and DM

SIESTA expands electronic states in localized atomic orbitals. Here $i,j$ label orbitals, and $\phi_{j\mathbf R}$ is orbital $j$ in the cell translated by $\mathbf R$:

| Object | Matrix element | Role |
|---|---|---|
| H | $H_{ij}(\mathbf R)=\langle\phi_{i0}|\hat H|\phi_{j\mathbf R}\rangle$ | Effective one-electron energies and couplings; units: eV |
| S | $S_{ij}(\mathbf R)=\langle\phi_{i0}|\phi_{j\mathbf R}\rangle$ | Orbital overlap; dimensionless |
| DM | $D_{ij}(\mathbf R)$ | Occupation-weighted coefficients of orbital pairs; dimensionless |

**H — Hamiltonian.** The converged Kohn–Sham Hamiltonian contains kinetic energy and the ionic, Hartree, and exchange–correlation potentials. Its diagonal describes same-orbital matrix elements; off-diagonal elements couple orbitals within and between cells. Band energies follow from solving the full H/S eigenvalue problem.

**S — overlap.** Atomic orbitals on different sites can overlap, so the basis is nonorthogonal and $S(\mathbf k)$ generally differs from the identity. S sets the normalization, $c^\dagger S c=1$, and appears in $Hc=\varepsilon Sc$.

**DM — density matrix.** It combines eigenstate coefficients with their occupations. At each k point,

$$D_{ij}(\mathbf k)=\sum_n f_{n\mathbf k}\,c_{in}(\mathbf k)c_{jn}^{*}(\mathbf k).$$

Here $c_{in}$ is orbital $i$'s coefficient in band $n$ and $f_{n\mathbf k}$ is its occupation, including both spins in this nonmagnetic calculation. The real-space coefficients follow from the k-point sum,

$$D_{ij}(\mathbf R)=\sum_{\mathbf k}w_{\mathbf k}
 e^{-i\mathbf k\cdot\mathbf R}D_{ij}(\mathbf k).$$

The stored coefficients weight orbital products to reconstruct $n(\mathbf r)$ in Section 6. In a nonorthogonal basis, $D_{ii}$ alone is not an orbital electron population; electron counting also uses S:

$$N_e=\sum_{\mathbf k}w_{\mathbf k}\operatorname{Tr}[D(\mathbf k)S(\mathbf k)].$$

`get_sile("RUN.fdf")` connects the inputs and outputs. H and S come from `BN.HSX`; DM comes from `BN.DM`. H carries the overlap needed for diagonalization, while `B.ion.xml` and `N.ion.xml` provide the orbital shapes needed for real-space projection.

**Energy reference.** For the SIESTA 5.4.2 HSX file used here, the [sisl 0.16.4 reader](https://github.com/zerothi/sisl/blob/v0.16.4/src/sisl/io/siesta/binaries.py#L1334-L1337) already shifts the Hamiltonian to the Fermi reference:

$$H=H_{\mathrm{KS}}-E_F S,\qquad \varepsilon_{n\mathbf k}=E_{n\mathbf k}-E_F.$$

Throughout the notebook, H denotes this shifted matrix and its eigenvalues are already $E-E_F$. `Ef_file` records the original SIESTA Fermi level for information only; use the returned energies directly.

**Matrix indices label orbitals.** This SZP basis has 9 orbitals per atom (one s, three p, and five polarization d), giving 36 per cell. `a2o(..., all=True)` maps each atom to its orbitals.


In [ ]:
H = fdf.read_hamiltonian(order=["HSX"])
S = fdf.read_overlap(order=["HSX"])
DM = fdf.read_density_matrix(order=["DM"])
Ef_file = si.get_sile("BN.HSX").read_fermi_level()  # Original SIESTA reference only.
geometry = H.geometry

print(f"{geometry.na} atoms, {geometry.no} orbitals")
print(f"Saved EF in the SIESTA reference: {Ef_file:.4f} eV; loaded H uses EF = 0")
for ia, atom in enumerate(geometry.atoms):
    print(f"atom {ia} ({atom.symbol}): orbitals {geometry.a2o(ia, all=True)}")

### Matrix dimensions and cell offsets

sisl stores neighboring-cell blocks side by side:

$$M=[M(\mathbf0)\;M(\mathbf R_1)\;\cdots],\qquad
M_{i,j+N_o b}=M_{ij}(\mathbf R_b).$$

Rows select the $N_o$ reference-cell orbitals; column `j + no * b` selects orbital `j` in cell-offset slot `b`. `nsc` describes the stored neighboring cells. For these inputs, $N_o=36$ and `nsc=[7, 7, 3]`, so a component has shape $36\times(36\cdot147)=36\times5292$.

The object shape adds a component axis: H stores Hamiltonian and overlap, S stores overlap only, and DM has density coefficients plus an overlap slot. Use `tocsr(dim=0)` for each physical matrix, and the HSX-derived S for overlap. `nsc` controls storage; `tile()` later enlarges the reference cell.


In [ ]:
no = geometry.no
print("nsc:", geometry.nsc, "→ stored cell offsets:", geometry.lattice.n_s)
print("First offsets:\n", geometry.lattice.sc_off[:5])
for name, matrix in [("H", H), ("S", S), ("DM", DM)]:
    print(f"{name}: object {matrix.shape}, component 0 {matrix.tocsr(dim=0).shape}")


### Matrix values with `imshow`

Display **H, S, and DM values**, including zeros. Rows run downward, as in the sparsity plot. The full arrays contain 147 cell blocks, so this wide view shows their layout; use the square blocks below to read individual elements.

The values span many orders of magnitude. `SymLogNorm` keeps zero white and distinguishes signs and weaker couplings: the scale is linear near zero and logarithmic outside ±0.001 of each matrix's maximum magnitude. The colorbar labels are the actual matrix values. The block plots below use linear color scales.


In [ ]:
from matplotlib.colors import SymLogNorm

matrices = [H, S, DM]
matrix_labels = ["H [eV]", "S", "DM"]
fig, axes = plt.subplots(3, 1, figsize=(14, 7), dpi=200, constrained_layout=True)
for ax, matrix, label in zip(axes, matrices, matrix_labels):
    values = matrix.tocsr(dim=0).toarray()
    limit = np.abs(values).max()
    norm = SymLogNorm(linthresh=0.001 * limit, vmin=-limit, vmax=limit)
    im = ax.imshow(values, origin="upper", aspect="auto", interpolation="nearest",
                   cmap="RdBu_r", norm=norm)
    ax.set(title=label, xlabel="j + no × cell-offset slot", ylabel="reference orbital i")
    fig.colorbar(im, ax=ax, ticks=[-limit, -limit / 100, 0, limit / 100, limit],
                 format="%.2g", label="symmetric log scale")
plt.show()


### Resolve individual cell blocks

`sc_index(isc)` finds the stored block for a lattice translation: `[0, 0, 0]` is the reference cell, and `[1, 0, 0]` is its neighbor along $\mathbf a_1$. Each block is $36\times36$; thin lines separate atoms. The two offsets share a color scale for each quantity.

For these Hermitian operators, $M(\mathbf R)=M(-\mathbf R)^\dagger$. A single neighbor block can therefore be asymmetric. Band energies come from H and S together, rather than from H's diagonal alone.


In [ ]:
offsets = [[0, 0, 0], [1, 0, 0]]
fig, axes = plt.subplots(2, 3, figsize=(12, 7), constrained_layout=True)
for col, (matrix, label) in enumerate(zip(matrices, matrix_labels)):
    blocks = []
    for isc in offsets:
        b = matrix.geometry.sc_index(isc)
        blocks.append(matrix.tocsr(dim=0)[:, b * no:(b + 1) * no].toarray())
    limit = max(np.abs(block).max() for block in blocks)
    for row, (isc, block) in enumerate(zip(offsets, blocks)):
        ax = axes[row, col]
        im = ax.imshow(block, origin="upper", interpolation="nearest",
                       cmap="RdBu_r", vmin=-limit, vmax=limit)
        for boundary in geometry.firsto[1:] - 0.5:
            ax.axhline(boundary, color="gray", lw=0.4)
            ax.axvline(boundary, color="gray", lw=0.4)
        ax.set(title=f"{label}, isc={isc}", xlabel="orbital j", ylabel="orbital i")
    fig.colorbar(im, ax=axes[:, col], shrink=0.8)
plt.show()


### Sparsity with `spy`

`spy` marks nonzero entries to show which orbitals connect. Use it alongside the value plots to see how localized orbitals make these matrices sparse.


In [ ]:
h_sparse = H.tocsr(dim=0)
fig, ax = plt.subplots(figsize=(12, 3), constrained_layout=True)
ax.spy(h_sparse, markersize=0.7, aspect="auto", color="tab:blue")
ax.xaxis.tick_bottom()
ax.set(xlabel="Orbital j across stored cells", ylabel="Reference-cell orbital i",
       title=f"H sparsity: {h_sparse.shape[0]} × {h_sparse.shape[1]}")
print(f"Nonzero entries: {h_sparse.count_nonzero() / np.prod(h_sparse.shape):.1%}")
plt.show()


## 4. Bloch matrices, bands, and DOS

### Why the matrix becomes square at each k

For a Bloch state with energy $\varepsilon=E-E_F$, coefficients in translated cells differ only by a phase:

$$c_{j\mathbf R}=e^{i\mathbf k\cdot\mathbf R}c_{j0}.$$

Insert this relation into the reference-cell equation,

$$\sum_{j,\mathbf R}H_{ij}(\mathbf R)c_{j\mathbf R}
=\varepsilon\sum_{j,\mathbf R}S_{ij}(\mathbf R)c_{j\mathbf R},$$

to obtain

$$\sum_j\underbrace{\left[\sum_{\mathbf R}e^{i\mathbf k\cdot\mathbf R}
H_{ij}(\mathbf R)\right]}_{H_{ij}(\mathbf k)}c_{j0}
=\varepsilon\sum_j\underbrace{\left[\sum_{\mathbf R}e^{i\mathbf k\cdot\mathbf R}
S_{ij}(\mathbf R)\right]}_{S_{ij}(\mathbf k)}c_{j0}.$$

The sum over cell offsets leaves one row and column for each reference-cell orbital:

$$\underbrace{[H(\mathbf0)\;H(\mathbf R_1)\;\cdots]}_{36\times5292}
\xrightarrow{\text{phase-weighted sum of 147 blocks}}
\underbrace{H(\mathbf k)}_{36\times36}.$$

**All 36 orbitals and their periodic couplings remain.** Translational symmetry relates neighboring-cell coefficients to those in the reference cell. Solving the resulting problem at each k gives 36 bands across the Brillouin zone.

### Build the Bloch sum explicitly

sisl takes fractional reciprocal coordinates $\mathbf q$. For an integer cell offset $\mathbf n$, the phase is $e^{2\pi i\mathbf q\cdot\mathbf n}$. This is the default lattice gauge used by `H.Hk()` and `H.Sk()`.

`blocks[i, b, j]` is $H_{ij}(\mathbf R_b)$. Multiply each block by its phase and use `sum(axis=1)` to sum the cell-offset axis. At K, the result should agree with `H.Hk()`.


In [ ]:
q = np.array([1/3, 1/3, 0])  # K in fractional reciprocal coordinates
blocks = H.tocsr(dim=0).toarray().reshape(no, -1, no)
phase = np.exp(2j * np.pi * (geometry.lattice.sc_off @ q))
Hq_from_blocks = (blocks * phase[None, :, None]).sum(axis=1)
Hq = H.Hk(k=q, format="array")
print("Stored component → H(k):", H.tocsr(dim=0).shape, "→", Hq.shape)
print("Bloch-sum difference [eV]:", np.max(np.abs(Hq_from_blocks - Hq)))


### Γ-point matrices

At Γ every phase equals 1, so **all neighboring-cell blocks still contribute**:

$$H(\Gamma)=\sum_{\mathbf R}H(\mathbf R)\ne H(\mathbf R=\mathbf0)\quad\text{in general}.$$

`H.Hk(k=[0, 0, 0])` and `H.Sk(k=[0, 0, 0])` return these sums. Plot their values below; the `[0, 0, 0]` blocks shown earlier contain only within-cell matrix elements.


In [ ]:
k = [0.0, 0.0, 0.0]  # Γ
Hk = H.Hk(k=k, format="array")
Sk = H.Sk(k=k, format="array")
fig, axes = plt.subplots(1, 2, figsize=(9, 4), constrained_layout=True)
for ax, matrix, title in zip(axes, [Hk, Sk], ["H(Γ) [eV]", "S(Γ)"]):
    limit = np.abs(matrix.real).max()
    im = ax.imshow(matrix.real, origin="upper", interpolation="nearest",
                   cmap="RdBu_r", vmin=-limit, vmax=limit)
    ax.set(title=title, xlabel="orbital j", ylabel="orbital i")
    fig.colorbar(im, ax=ax)
plt.show()


### Eigenvalues in a nonorthogonal basis

The orbital coefficients satisfy the generalized eigenvalue problem and normalization:

$$H(\mathbf k)c_{n\mathbf k}=\varepsilon_{n\mathbf k}S(\mathbf k)c_{n\mathbf k},
\qquad c_{n\mathbf k}^{\dagger}S(\mathbf k)c_{n\mathbf k}=1.$$

`H.eigh()` includes S and returns 36 eigenvalues per k point. Using `np.linalg.eigh(Hk)` would assume an orthonormal basis. The returned $\varepsilon_{n\mathbf k}$ is already $E-E_F$, so no further Fermi-level subtraction is needed.


In [ ]:
eigenvalues = H.eigh(k=k)
print("H(Γ), S(Γ):", Hk.shape, Sk.shape)
print("Number of bands:", len(eigenvalues))
print("Bands 7–10 at Γ, E − EF [eV]:", np.round(eigenvalues[6:10], 3))


### Band structure

`BandStructure` samples Γ–M–K–Γ, and `apply.array.eigh()` solves for the bands along this path. The saved H is sufficient; no new SCF run is needed.


In [ ]:
points = np.array([[0, 0, 0], [0.5, 0, 0], [1/3, 1/3, 0], [0, 0, 0]])
names = ["Γ", "M", "K", "Γ"]
bs = si.BandStructure(H, points, 121, names=names)
bands = bs.apply.array.eigh()  # Already E - EF.
x = bs.lineark()
ticks, tick_labels = bs.lineartick()

fig, ax = plt.subplots(figsize=(7, 4), constrained_layout=True)
ax.plot(x, bands, color="tab:blue", lw=0.8)
ax.set(xticks=ticks, xticklabels=tick_labels, xlim=(x[0], x[-1]), ylim=(-15, 15),
       ylabel="E − EF [eV]", title="BN band structure")
ax.axhline(0, color="gray", ls=":")
ax.grid(axis="x")
plt.show()


### Density of states

DOS counts states over a Brillouin-zone mesh:

$$g(\varepsilon)=\sum_{\mathbf k}w_{\mathbf k}\sum_n\delta(\varepsilon-\varepsilon_{n\mathbf k}).$$

Following the [official graphene tutorial](https://zerothi.github.io/sisl/tutorials/tutorial_siesta_2.html#calculating-dos-and-pdos), use `MonkhorstPack` and average `eigenvalue().DOS()`. Eigenvectors are unnecessary for total DOS. A denser 18×18×6 post-processing mesh reduces the isolated sampling peaks of the SCF mesh; the default Gaussian width is 0.1 eV. This is an illustrative mesh, not a convergence study.

For this nonmagnetic H, the result is **per spin channel**. The eigenvalues and `energy` grid are both relative to EF = 0, so pass `energy` directly to `DOS()`.


In [ ]:
bz = si.MonkhorstPack(H, [18, 18, 6])
energy = np.linspace(-15, 15, 1001)
dos = bz.apply.average.eigenvalue(wrap=lambda ev: ev.DOS(energy))

fig, ax = plt.subplots(figsize=(7, 3), constrained_layout=True)
ax.plot(energy, dos)
ax.axvline(0, color="gray", ls=":")
ax.set(xlabel="E − EF [eV]", ylabel="DOS [states/eV/cell/spin]",
       xlim=(energy[0], energy[-1]), ylim=(0, None))
plt.show()


## 5. Supercell bands on the same path

`H.tile(2, axis=0)` doubles the first lattice vector and repeats H and S. The cell has 72 orbitals and 72 bands, with a halved first reciprocal vector:

$$\mathbf A_1=2\mathbf a_1,\qquad \mathbf B_1=\tfrac12\mathbf b_1.$$

Keep the **same physical Γ–M–K–Γ path** used above. Original-cell fractional coordinates $\mathbf q$ become supercell coordinates $\mathbf Q$:

$$\mathbf k=\sum_i q_i\mathbf b_i=\sum_i Q_i\mathbf B_i,
\qquad \mathbf Q=(2q_1,q_2,q_3).$$

Thus `points * [2, 1, 1]` follows the same path. The Γ/M/K labels refer to the original cell; coordinates can extend beyond the supercell's first Brillouin zone.

Band folding places the original $\mathbf q$ and $\mathbf q+(1/2,0,0)$ branches at the same supercell wavevector:

$$\{E_m^{\mathrm{SC}}(\mathbf Q)\}
=\{E_n(\mathbf q)\}\cup\{E_n(\mathbf q+(1/2,0,0))\}.$$

The plot below shows the supercell bands along this path.


In [ ]:
H2 = H.tile(2, axis=0)
points2 = points * [2, 1, 1]
bs2 = si.BandStructure(H2, points2, 121, names=names)
bands2 = bs2.apply.array.eigh()  # Same E - EF reference as H.
x2 = bs2.lineark()
ticks2, tick_labels2 = bs2.lineartick()
print(f"Orbitals per cell: {H.no} → {H2.no}")

fig, ax = plt.subplots(figsize=(7, 4), constrained_layout=True)
ax.plot(x2, bands2, color="tab:blue", lw=0.8)
ax.set(xticks=ticks2, xticklabels=tick_labels2, xlim=(x2[0], x2[-1]), ylim=(-15, 15),
       ylabel="E − EF [eV]", title="2× BN supercell: folded bands",
       xlabel="Γ–M–K–Γ path of the original cell")
ax.axhline(0, color="gray", ls=":")
ax.grid(axis="x")
plt.show()


## 6. Electron density in real space

DM weights pairs of numerical orbitals. For the real atomic orbitals used here,

$$n(\mathbf r)=\sum_{ij}\sum_{\mathbf T,\mathbf R}
D_{ij}(\mathbf R)\phi_i(\mathbf r-\mathbf T)
\phi_j(\mathbf r-\mathbf T-\mathbf R).$$

$\mathbf T$ runs over translated reference cells, and $\mathbf R$ is the relative offset stored in DM. `DM.density(rho)` evaluates this sum on a spatial grid using the basis from `B.ion.xml` and `N.ion.xml`.

Create a fresh grid with target spacing 0.08 Å, since the projection adds to its contents. The nonmagnetic DM already includes both spin occupations. Its density integrates to the 16 valence electrons in the four-atom cell:

$$N_e=\int_{\mathrm{cell}}n(\mathbf r)\,d^3r
\approx\sum_g n_g\,\Delta V=2\times3+2\times5=16.$$


In [ ]:
rho = si.Grid(0.08, geometry=DM.geometry)
DM.density(rho)
print("Grid shape:", rho.shape)
print(f"Electron count: {rho.grid.sum() * rho.dvolume:.3f} (expected: 16)")


### 2D electron-density map

Use the 2D workflow from the [official graphene tutorial](https://zerothi.github.io/sisl/tutorials/tutorial_siesta_2.html#hamiltonian-eigenstates): locate the slice with `Grid.index()`, convert grid indices to Cartesian x/y using `dcell`, and draw filled contours with atomic positions.

Repeat the density over a **2×2 in-plane area** with `Grid.tile()` to show periodic structure. The electron count above belongs to the original cell. The slice is about **0.5 Å above the first BN layer**; the markers project that layer's B/N positions onto the plot. The same coordinates are used for the Γ-point wavefunction below.


In [ ]:
rho_view = rho.tile(2, axis=0).tile(2, axis=1)
view_geometry = rho_view.geometry
layer_z = geometry.xyz[:, 2].min()
iz = rho_view.index([0, 0, layer_z + 0.5], axis=2)
z_actual = rho_view.index2xyz([0, 0, iz])[2]
i, j = np.mgrid[:rho_view.shape[0], :rho_view.shape[1]]
x_grid = i * rho_view.dcell[0, 0] + j * rho_view.dcell[1, 0]
y_grid = i * rho_view.dcell[0, 1] + j * rho_view.dcell[1, 1]
layer_atoms = np.flatnonzero(np.isclose(view_geometry.xyz[:, 2], layer_z))

def mark_layer(ax):
    for symbol, color in [("B", "hotpink"), ("N", "limegreen")]:
        atoms = [ia for ia in layer_atoms if view_geometry.atoms[ia].symbol == symbol]
        xy = view_geometry.xyz[atoms, :2]
        ax.scatter(xy[:, 0], xy[:, 1], s=28, c=color, edgecolors="black",
                   linewidths=0.6, label=symbol, zorder=3)
    ax.set(xlabel="x [Å]", ylabel="y [Å]", aspect="equal")
    ax.legend(loc="upper right", fontsize=8)

fig, ax = plt.subplots(figsize=(6, 5), constrained_layout=True)
values = rho_view.grid[:, :, iz]
im = ax.contourf(x_grid, y_grid, values, levels=np.linspace(0, values.max(), 41), cmap="magma")
mark_layer(ax)
ax.set_title(f"Electron density: 2×2 view, z = {z_actual:.2f} Å")
fig.colorbar(im, ax=ax, ticks=np.linspace(0, values.max(), 5),
             format="%.2f", label="electrons / Å³", shrink=0.8)
plt.show()


## 7. Wavefunction and probability density

An eigenstate's coefficients give its real-space wavefunction:

$$\psi_{n\mathbf k}(\mathbf r)=\sum_{i,\mathbf R}
c_{in}(\mathbf k)e^{i\mathbf k\cdot\mathbf R}\phi_i(\mathbf r-\mathbf R),
\qquad p_{n\mathbf k}(\mathbf r)=|\psi_{n\mathbf k}(\mathbf r)|^2.$$

`H.eigenstate(k=...)` solves for the states at one k point; `.sub(band_index)` selects one band. At Γ, index 7 is the eighth band and the highest occupied band there for this 16-electron nonmagnetic cell. Index 8 selects the first unoccupied band.

`state.wavefunction(psi)` projects that state onto a fresh grid. Its probability integrates to 1 per cell, while the total density includes all occupied states:

$$\int_{\mathrm{cell}}|\psi_{n\mathbf k}(\mathbf r)|^2\,d^3r=1,
\qquad n(\mathbf r)=\sum_{n\mathbf k}w_{\mathbf k}f_{n\mathbf k}
|\psi_{n\mathbf k}(\mathbf r)|^2.$$

Here $f_{n\mathbf k}$ includes spin occupations. Keep Γ for this example; `band_index` controls which state's spatial structure is displayed.


In [ ]:
k_wave = [0.0, 0.0, 0.0]  # Γ
band_index = 7             # Eighth band; zero-based index
state = H.eigenstate(k=k_wave).sub(band_index)
psi = si.Grid(rho.shape, geometry=H.geometry, dtype=complex)
state.wavefunction(psi)
probability = np.abs(psi.grid)**2
print(f"Γ, band {band_index + 1}: E − EF = {state.eig[0]:.3f} eV")
print(f"∫ |ψ|² dV = {probability.sum() * psi.dvolume:.3f} (expected: 1)")


### 2D wavefunction and probability maps

At Γ, the eigenvectors can be real and the wavefunction is cell-periodic, so `Grid.tile()` repeats its values over the same 2×2 view. Plot **Re $\psi$** with a symmetric color scale to show signs and nodes, and **$|\psi|^2$** with a positive scale to show probability. The wavefunction's overall sign is arbitrary; red/blue indicate amplitude signs. For degenerate states, the individual shape also depends on the eigenvector combination returned by the solver.


In [ ]:
psi_view = psi.tile(2, axis=0).tile(2, axis=1)
wave_slice = psi_view.grid[:, :, iz].real
probability_slice = np.abs(psi_view.grid[:, :, iz])**2
amplitude = np.abs(wave_slice).max()
fig, axes = plt.subplots(1, 2, figsize=(11, 5), constrained_layout=True)
for ax, values, title, cmap, levels, unit in zip(
    axes, [wave_slice, probability_slice], ["Re ψ", "|ψ|²"],
    ["RdBu_r", "magma"],
    [np.linspace(-amplitude, amplitude, 41), np.linspace(0, probability_slice.max(), 41)],
    [r"amplitude [Å$^{-3/2}$]", r"probability density [Å$^{-3}$]"],
):
    im = ax.contourf(x_grid, y_grid, values, levels=levels, cmap=cmap)
    mark_layer(ax)
    ax.set_title(title)
    fig.colorbar(im, ax=ax, ticks=np.linspace(levels[0], levels[-1], 5),
                 format="%.2f", label=unit, shrink=0.8)
fig.suptitle(f"Γ, band {band_index + 1}: 2×2 view, z = {z_actual:.2f} Å")
plt.show()


## Try it

- Select band index 8 and rerun the wavefunction and map cells. How do the nodes and probability distribution change?
- Change the selected neighbor offset to `[-1, 0, 0]`. How are its matrix elements related to the `[1, 0, 0]` block?
- Tile three cells along the first axis and use `points * [3, 1, 1]`. How many bands appear on the same path?

## References

- [Official SIESTA/graphene tutorial: DOS, bands, and 2D wavefunctions](https://zerothi.github.io/sisl/tutorials/tutorial_siesta_2.html)
- [Official SIESTA/H₂O tutorial: atomic orbitals and eigenstates](https://zerothi.github.io/sisl/tutorials/tutorial_siesta_1.html)

- [Hamiltonian: Hk, Sk, eigh, tile](https://zerothi.github.io/sisl/api/generated/sisl.physics.Hamiltonian.html)
- [BandStructure: reciprocal-space paths](https://zerothi.github.io/sisl/api/generated/sisl.physics.BandStructure.html)
- [DensityMatrix: density](https://zerothi.github.io/sisl/api/generated/sisl.physics.DensityMatrix.html)
- [Wavefunction: real-space projection](https://zerothi.github.io/sisl/api/generated/sisl.physics.electron.wavefunction.html)
- [sisl workshop: matrix sparsity with spy](https://github.com/zerothi/ts-tbt-sisl-tutorial/blob/master/TB_02/run.ipynb)
